In [ ]:
import pandas as pd

In [ ]:
df = pd.read_excel('/content/Motor Trend Car Road Tests (2).xlsx')
display(df.head())

,model,mpg,cyl,disp,hp,drat,wt,qsec,vs,am,gear,carb
0,Mazda RX4,21.0,6,160.0,110,3.90,2.620,16.46,0,1,4,4
1,Mazda RX4 Wag,21.0,6,160.0,110,3.90,2.875,17.02,0,1,4,4
2,Datsun 710,22.8,4,108.0,93,3.85,2.320,18.61,1,1,4,1
3,Hornet 4 Drive,21.4,6,258.0,110,3.08,3.215,19.44,1,0,3,1
4,Hornet Sportabout,18.7,8,360.0,175,3.15,3.440,17.02,0,0,3,2


In [ ]:
print(f"Renglones: {df.shape[0]} y columnas: {df.shape[1]}")

Renglones: 32 y columnas: 12


In [ ]:
import statsmodels.formula.api as smf
import statsmodels.api as sm

# Define variable (y)
y = df['mpg']

# Define variables (X)
X = df[['hp', 'qsec']]

# Add a constant (intercept) to the independent variables
X = sm.add_constant(X)

# Fit the OLS model
model = sm.OLS(y, X).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                    mpg   R-squared:                       0.637
Model:                            OLS   Adj. R-squared:                  0.612
Method:                 Least Squares   F-statistic:                     25.43
Date:                Mon, 04 May 2026   Prob (F-statistic):           4.18e-07
Time:                        22:38:54   Log-Likelihood:                -86.170
No. Observations:                  32   AIC:                             178.3
Df Residuals:                      29   BIC:                             182.7
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         48.3237     11.103      4.352      0.0

In [ ]:
import scipy.stats as st
import statsmodels.api as sm

# Bootstraps
n_bootstraps = 1000

bootstrap_coefs = []

for i in range(n_bootstraps):
    # Reemplazo
    bootstrap_sample = df.sample(n=len(df), replace=True)

    # Definir y
    y_bootstrap = bootstrap_sample['mpg']

    # Definir X
    X_bootstrap = bootstrap_sample[['hp', 'qsec']]

    # Intercepto
    X_bootstrap = sm.add_constant(X_bootstrap)

    # Fit
    bootstrap_model = sm.OLS(y_bootstrap, X_bootstrap).fit()

    # Guarda los coeficientes
    bootstrap_coefs.append(bootstrap_model.params)

# Convertir a DataFrame
bootstrap_coefs_df = pd.DataFrame(bootstrap_coefs)

# Mean de los coeficientes
mean_coefs = bootstrap_coefs_df.mean()
print("\n Media de los coeficientes Bootstrap (Bx):\n", mean_coefs)

# DE
std_dev_coeffs = bootstrap_coefs_df.std()
print("\n Desviación Estandar de Bootstrap (Desviacionb):\n", std_dev_coeffs)

# IC
lower_bound = mean_coefs - (2 * std_dev_coeffs)
upper_bound = mean_coefs + (2 * std_dev_coeffs)

confidence_intervals = pd.DataFrame({
    'Bajo IC': lower_bound,
    'Arriba IC': upper_bound
})
print("\n95% Bootstrap Aproximación Normal:\n", confidence_intervals)


 Media de los coeficientes Bootstrap (Bx):
 const    49.360543
hp       -0.086820
qsec     -0.930814
dtype: float64

 Desviación Estandar de Bootstrap (Desviacionb):
 const    10.755277
hp        0.016151
qsec      0.521270
dtype: float64

95% Bootstrap Aproximación Normal:
          Bajo IC  Arriba IC
const  27.849990  70.871097
hp     -0.119122  -0.054518
qsec   -1.973353   0.111725


## Comparación de Resultados: Regresión vs. Bootstrap


**1. Coeficientes:**
*   **Intercept (const):**
    *   OLS: `48.3237`
    *   Bootstrap Media (Bx): `49.5377`
*   **hp:**
    *   OLS: `-0.0846`
    *   Bootstrap Media (Bx): `-0.0867`
*   **qsec:**
    *   OLS: `-0.8866`
    *   Bootstrap Media (Bx): `-0.9427`

**2. Errores Estándar / Desviaciones Estándar:**
*   **Intercept (const):**
    *   OLS Error Estándar: `11.103`
    *   Bootstrap Desviación Estándar (Desviacionb): `11.0735`
*   **hp:**
    *   OLS Error Estándar: `0.014`
    *   Bootstrap Desviación Estándar (Desviacionb): `0.0168`
*   **qsec:**
    *   OLS Error Estándar: `0.535`
    *   Bootstrap Desviación Estándar (Desviacionb): `0.5249`

**3. Intervalos de Confianza del 95%:**
*   **Intercept (const):**
    *   OLS: ´[25.615, 71.033]´
    *   Bootstrap (Aprox. Normal): `[27.3908, 71.6847]`
*   **hp:**
    *   OLS: `[-0.113, -0.056]`
    *   Bootstrap (Aprox. Normal): `[-0.1203, -0.0531]`
*   **qsec:**
    *   OLS: `[-1.980, 0.207]`
    *   Bootstrap (Aprox. Normal): `[-1.9926, 0.1071]`

**Resumen de la Comparación:**
*   **Coeficientes:** Los coeficientes de ambos métodos son bastante iguales, lo que es bueno.
*   **Errores Estándar/Desviaciones:** Las desviaciones estándar del análisis bootstrap son generalmente cercanas a los errores estándar del modelo OLS, es bastante bueno de igual manera
*   **Intervalos de Confianza:** Los intervalos de confianza del 95% también son muy similares entre los dos enfoques, lo que refuerza lo que se dijo antes

## Aggregating

In [ ]:
import random
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
import pandas as pd

# número de simulaciones
n_simulaciones = 1000

# Quitar la variable "y" (mpg) y las columnas que no tienen número ('model')
#df.drop(columns=[tales])
columnas_nuevas = df.select_dtypes(include=['number']).columns.drop('mpg').tolist()

# Listas para guardar los coeficientes
#sim_coefs = []
sim_models = [] # Lista para guardar los modelos
sim_random_columns_ = [] # Nueva lista para guardar las columnas aleatorias
# División train-test (50/50)
df_train, df_test = train_test_split(df, test_size=0.5, random_state=1)
print(f"Simulaciones {n_simulaciones} ...")
import numpy as np
for i in range(n_simulaciones):
    # Seleccionar 3 columnas predictoras al azar
    #np.random.choice(lista, cuántas, replace=False)
    random_columnas = np.random.choice(columnas_nuevas, 3, replace=False)
    #random_columnas = random.sample(columnas_nuevas, 3)
    sim_random_columns_.append(random_columnas) # Guardar las columnas usadas

    # Variable dependiente (y_train) y las variables independientes (X_train)
    y_train = df_train['mpg']
    X_train = df_train[random_columnas]

    # Ajustar el modelo sklearn LinearRegression
    modelo = LinearRegression()
    modelo.fit(X_train, y_train)

    sim_models.append(modelo) # Guardar el modelo ajustado


Simulaciones 1000 ...


In [ ]:
columnas_nuevas

['cyl', 'disp', 'hp', 'drat', 'wt', 'qsec', 'vs', 'am', 'gear', 'carb']

In [15]:
import numpy as np

# Lista para guardar las predicciones de cada modelo
sim_predicciones = []

for i, model in enumerate(sim_models):
    # Obtener las columnas usadas para este modelo específico
    random_columnas = sim_random_columns_[i]

    # Predecir en el conjunto de prueba (df_test) usando las mismas columnas
    X_test = df_test[random_columnas]
    predicciones = model.predict(X_test)
    sim_predicciones.append(predicciones)

# Convertir la lista de predicciones a un array de NumPy
sim_predicciones_array = np.array(sim_predicciones)

# Calcular la predicción agregada (el promedio de todas las predicciones)
aggregated_predicciones = np.mean(sim_predicciones_array, axis=0)

# Añadir las predicciones agregadas al DataFrame de prueba para comparar
df_test['aggregated_predicciones'] = aggregated_predicciones

# Mostrar las primeras filas del DataFrame de prueba con las predicciones
display(df_test[['mpg', 'aggregated_predicciones']].head())

,mpg,aggregated_predicciones
27,30.4,24.321851
3,21.4,19.315119
22,15.2,16.681200
18,30.4,25.845286
23,13.3,15.218822


## Random Forest

### Optimización Bayesiana para Hiperparámetros de Random Forest

Dado que las búsquedas anteriores no han alcanzado el R2 deseado, implementaremos la Optimización Bayesiana. Esta técnica es más eficiente que `GridSearchCV` para explorar espacios de hiperparámetros grandes, ya que utiliza un modelo probabilístico para predecir qué combinaciones de parámetros tienen más probabilidades de dar un buen resultado.

Utilizaremos la biblioteca `bayesian-optimization` para encontrar los hiperparámetros óptimos para nuestro `RandomForestRegressor`.

In [ ]:
import warnings
from sklearn.metrics import make_scorer, r2_score
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold # Import KFold

try:
    import bayes_opt
    from bayes_opt import BayesianOptimization
except ImportError:
    print("Installing bayesian-optimization...")
    !pip install bayesian-optimization
    from bayes_opt import BayesianOptimization

warnings.filterwarnings('ignore') # Ignorar warnings para una salida más limpia

# Asegurémonos de que X e y estén definidas
X = df.drop(columns=['mpg', 'model'])
y = df['mpg']

# Initialize KFold outside the function to avoid re-initializing it in each iteration
kf = KFold(n_splits=10, shuffle=True, random_state=42)

# Definir la función objetivo para la optimización bayesiana
# Esta función recibe los hiperparámetros a optimizar
def rf_bayesian_optimization(n_estimators, max_depth, min_samples_split, min_samples_leaf, max_features):
    # Convertir los floats a enteros donde sea necesario
    n_estimators = int(n_estimators)
    max_depth = int(max_depth) if max_depth > 0 else None # None si max_depth <= 0
    min_samples_split = int(min_samples_split)
    min_samples_leaf = int(min_samples_leaf)
    # max_features puede ser un float (proporción) o una cadena (sqrt, log2)
    # Aseguramos que sea una de las opciones válidas para RandomForestRegressor
    if max_features <= 0.33:
        max_features_val = 'sqrt' # Aprox 1/3 de features
    elif max_features <= 0.66:
        max_features_val = 'log2' # Aprox 2/3 de features
    else:
        max_features_val = 1.0 # Todas las features

    model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features_val,
        random_state=42, # Para reproducibilidad
        n_jobs=-1 # Usar todos los cores disponibles
    )

    # Usar cross_val_score con KFold para evaluar el modelo
    # El scoring es el R2, y bayes_opt busca minimizar, por lo que devolvemos -R2
    score = cross_val_score(model, X, y, cv=kf, scoring='r2', n_jobs=-1).mean()

    return score

# Definir el espacio de búsqueda para los hiperparámetros
pbounds = {
    'n_estimators': (50, 1000),
    'max_depth': (1, 30), # Usaremos 1 para que sea None si es 0, y 30 como un limite superior razonable
    'min_samples_split': (2, 20),
    'min_samples_leaf': (1, 10),
    'max_features': (0.1, 1.0) # Se mapeará a 'sqrt', 'log2' o 1.0
}

# Inicializar el optimizador bayesiano
optimizer = BayesianOptimization(
    f=rf_bayesian_optimization,
    pbounds=pbounds,
    random_state=42,
    verbose=2 # Verbose = 1 prints a summary every step, verbose = 0 prints nothing
)


# Ejecutar la optimización
# n_iter es el número de iteraciones de optimización (evaluaciones de la función objetivo)
# init_points es el número de pasos de exploración aleatoria antes de usar el modelo bayesiano
optimizer.maximize(
    init_points=10,
    n_iter=50
)

# Obtener los mejores parámetros encontrados
best_params_bayes_opt = optimizer.max['params']
# El R2 score es el valor máximo (negativo de la función objetivo)
best_r2_bayes_opt = optimizer.max['target']

print(f"\nMejores hiperparámetros  por Optimización Bayesiana: {best_params_bayes_opt}")
print(f"Mejor R2 con Optimización Bayesiana: {best_r2_bayes_opt:.4f}")

# Opcional: Entrenar el modelo final con los mejores parámetros encontrados en el dataset completo
# Asegurarse de convertir los tipos de nuevo
final_n_estimators = int(best_params_bayes_opt['n_estimators'])
final_max_depth = int(best_params_bayes_opt['max_depth']) if best_params_bayes_opt['max_depth'] > 0 else None
final_min_samples_split = int(best_params_bayes_opt['min_samples_split'])
final_min_samples_leaf = int(best_params_bayes_opt['min_samples_leaf'])

final_max_features_val = 'sqrt'
if best_params_bayes_opt['max_features'] <= 0.33:
    final_max_features_val = 'sqrt'
elif best_params_bayes_opt['max_features'] <= 0.66:
    final_max_features_val = 'log2'
else:
    final_max_features_val = 1.0

final_model_rf = RandomForestRegressor(
    n_estimators=final_n_estimators,
    max_depth=final_max_depth,
    min_samples_split=final_min_samples_split,
    min_samples_leaf=final_min_samples_leaf,
    max_features=final_max_features_val,
    random_state=42,
    n_jobs=-1
)

final_model_rf.fit(X, y)

print("\nModelo final.")

print(f" R2: {best_r2_bayes_opt:.4f}")

|   iter    |  target   | n_esti... | max_depth | min_sa... | min_sa... | max_fe... |
-------------------------------------------------------------------------------------
| 1         | 0.3893677 | 405.81311 | 28.570714 | 15.175890 | 6.3879263 | 0.2404167 |
| 2         | 0.2361196 | 198.19479 | 2.6844247 | 17.591170 | 6.4100351 | 0.7372653 |
| 3         | 0.3821886 | 69.555269 | 29.127385 | 16.983967 | 2.9110519 | 0.2636424 |
| 4         | 0.4580634 | 224.23428 | 9.8230250 | 11.445615 | 4.8875051 | 0.3621062 |
| 5         | 0.4789689 | 631.26024 | 5.0453219 | 7.2586036 | 4.2972565 | 0.5104629 |
| 6         | 0.4110791 | 795.91716 | 6.7905396 | 11.256219 | 6.3317311 | 0.1418053 |
| 7         | 0.0859182 | 627.16760 | 5.9451995 | 3.1709286 | 9.5399698 | 0.9690688 |
| 8         | 0.3755113 | 817.97748 | 9.8337993 | 3.7580980 | 7.1580972 | 0.4961372 |
| 9         | 0.1267958 | 165.93632 | 15.360130 | 2.6189933 | 9.1838836 | 0.3329019 |
| 10        | 0.4275059 | 679.39617 | 10.039621 | 11.3

## Boosting

In [24]:
from sklearn.ensemble import GradientBoostingRegressor


# Definir la función objetivo para la optimización bayesiana de Gradient Boosting
def gbr_bayesian_optimization(n_estimators, max_depth, max_leaf_nodes, learning_rate, subsample):
    # Convertir los floats a enteros donde sea necesario
    n_estimators = int(n_estimators)
    max_depth = int(max_depth)
    max_leaf_nodes = int(max_leaf_nodes) if max_leaf_nodes > 0 else None # None si max_leaf_nodes <= 0

    model = GradientBoostingRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        max_leaf_nodes=max_leaf_nodes,
        learning_rate=learning_rate,
        subsample=subsample,
        random_state=42 # Para reproducibilidad
    )

    # Usar cross_val_score con KFold para evaluar el modelo
    score = cross_val_score(model, X, y, cv=kf, scoring='r2', n_jobs=-1).mean()

    return score

# Definir el espacio de búsqueda para los hiperparámetros
pbounds_gbr = {
    'n_estimators': (50, 1000),
    'max_depth': (1, 10),
    'max_leaf_nodes': (2, 32), # Un rango razonable para max_leaf_nodes
    'learning_rate': (0.01, 0.3),
    'subsample': (0.6, 1.0)
}

# Inicializar el optimizador bayesiano para Gradient Boosting
optimizer_gbr = BayesianOptimization(
    f=gbr_bayesian_optimization,
    pbounds=pbounds_gbr,
    random_state=42,
    verbose=2
)

print("Iniciando la Optimización Bayesiana para Gradient Boosting... ")

# Ejecutar la optimización
optimizer_gbr.maximize(
    init_points=10,
    n_iter=50
)

print("Optimización Bayesiana para Gradient Boosting completada.")

# Obtener los mejores parámetros encontrados
best_params_gbr_bayes_opt = optimizer_gbr.max['params']
best_r2_gbr_bayes_opt = optimizer_gbr.max['target']

print(f"\nMejores hiperparámetros encontrados por Optimización Bayesiana (Gradient Boosting): {best_params_gbr_bayes_opt}")
print(f"Mejor R2 score obtenido con Optimización Bayesiana (Gradient Boosting): {best_r2_gbr_bayes_opt:.4f}")

# Entrenar el modelo final con los mejores parámetros encontrados
final_n_estimators_gbr = int(best_params_gbr_bayes_opt['n_estimators'])
final_max_depth_gbr = int(best_params_gbr_bayes_opt['max_depth'])
final_max_leaf_nodes_gbr = int(best_params_gbr_bayes_opt['max_leaf_nodes']) if best_params_gbr_bayes_opt['max_leaf_nodes'] > 0 else None
final_learning_rate_gbr = best_params_gbr_bayes_opt['learning_rate']
final_subsample_gbr = best_params_gbr_bayes_opt['subsample']

final_model_gbr = GradientBoostingRegressor(
    n_estimators=final_n_estimators_gbr,
    max_depth=final_max_depth_gbr,
    max_leaf_nodes=final_max_leaf_nodes_gbr,
    learning_rate=final_learning_rate_gbr,
    subsample=final_subsample_gbr,
    random_state=42
)

final_model_gbr.fit(X, y)

print("\nModelo final de Gradient Boosting entrenado con los mejores hiperparámetros.")
print(f"El R2 final obtenido con la mejor configuración de hiperparámetros (mediante validación cruzada K-Fold) es: {best_r2_gbr_bayes_opt:.4f}")

Iniciando la Optimización Bayesiana para Gradient Boosting... Esto puede tomar un tiempo considerable.
|   iter    |  target   | n_esti... | max_depth | max_le... | learni... | subsample |
-------------------------------------------------------------------------------------
| 1         | 0.1672763 | 405.81311 | 9.5564287 | 23.959818 | 0.1836109 | 0.6624074 |
| 2         | 0.4290044 | 198.19479 | 1.5227525 | 27.985284 | 0.1843233 | 0.8832290 |
| 3         | 0.2741051 | 69.555269 | 9.7291886 | 26.973279 | 0.0715783 | 0.6727299 |
| 4         | 0.2914595 | 224.23428 | 3.7381801 | 17.742692 | 0.1352640 | 0.7164916 |
| 5         | 0.5327960 | 631.26024 | 2.2554447 | 10.764339 | 0.1162449 | 0.7824279 |
| 6         | 0.5179314 | 795.91716 | 2.7970640 | 17.427033 | 0.1818002 | 0.6185801 |
| 7         | 0.3691530 | 627.16760 | 2.5347171 | 3.9515477 | 0.2851768 | 0.9862528 |
| 8         | 0.4828210 | 817.97748 | 3.7415239 | 4.9301634 | 0.2084275 | 0.7760609 |
| 9         | 0.3782748 | 165.93632 |